# Basic pipeline optimization

This example shows how to tune a parameter inside a simple PipeOptz pipeline. The pipeline has one node, `Multiply`, which multiplies each input value by a configurable `factor`.

We start with `factor=1`, then register `Multiply.factor` as an integer parameter that the optimizer is allowed to change. The small training dataset follows the rule `expected = value * 3`, so grid search should discover that `factor=3` gives the lowest error.

The goal is to demonstrate the full optimization loop on a minimal example: define a pipeline, choose which fixed parameter can vary, provide input/output examples, run the optimizer, and inspect the best parameter value it found.

In [1]:
from pipeoptz import IntParameter, Node, Pipeline, PipelineOptimizer

Define the node function and the loss function minimized by the optimizer.

In [2]:
def multiply(value, factor):
    return value * factor


def absolute_error(actual, expected):
    return abs(actual - expected)

Build a pipeline with a fixed `factor`. The optimizer will update this parameter while searching.

In [3]:
pipeline = Pipeline("Multiplier optimization")
pipeline.add_node(
    Node(
        node_id="Multiply",
        func=multiply,
        fixed_params={"factor": 1},
    ),
    predecessors={"value": "run_params:value"},
)

Register `Multiply.factor` as an integer search parameter ranging from 0 to 5.

In [4]:
optimizer = PipelineOptimizer(pipeline, absolute_error)
optimizer.add_param(
    IntParameter(
        node_id="Multiply",
        param_name="factor",
        min_value=0,
        max_value=5,
    )
)

Fit the examples `1 → 3`, `2 → 6`, and `3 → 9`. Grid search evaluates every possible factor.

In [5]:
best_params, loss_log = optimizer.optimize(
    X=[{"value": 1}, {"value": 2}, {"value": 3}],
    y=[3, 6, 9],
    method="GS",
    max_combinations=6,
    param_sampling=7,
)

print("Best parameters:", best_params)
print("Best loss:", loss_log[-1])

Best parameters: {'Multiply.factor': 3}
Best loss: 0.0


The best parameter is left on the pipeline, so it can immediately be used for inference.

In [6]:
last_node, outputs, _ = pipeline.run({"value": 4})
print("Prediction for 4:", outputs[last_node])

Prediction for 4: 12
